In [ ]:
import os
import glob
import json
import time
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from openai import OpenAI



client = OpenAI(
    api_key="API",
    base_url="https://api.groq.com/openai/v1"
)


# testing 200 sentences to get a solid baseline without taking all day
SAMPLE_SIZE = 200 


print("Loading data... (this might take a few seconds)")

IDtoLabel_UK = {0: 'O', 1: 'B-Person', 2: 'I-Person', 3: 'B-Organization', 4: 'I-Organization', 5: 'B-Location', 6: 'I-Location', 7: 'B-Misc',  8: 'I-Misc'}
LabelToID = {'O': 0, 'B-Date': 1, 'I-Date': 2, 'B-Person': 3, 'I-Person': 4, 'B-Location': 5, 'I-Location': 6, 'B-Facility': 7, 'I-Facility': 8, 'B-Organization': 9, 'I-Organization': 10, 'B-Misc': 11, 'I-Misc': 12, 'B-Money': 13, 'I-Money': 14, 'B-NORP': 15, 'I-NORP': 16, 'B-Product': 17, 'I-Product': 18}
id2label = {v: k for k, v in LabelToID.items()}

# English Data
conll_data = load_dataset("BramVanroy/conll2003")
testDataUK = conll_data['test'].to_pandas().drop(columns=['pos_tags', 'chunk_tags', 'id'])
testDataUK['ner_tags'] = testDataUK['ner_tags'].apply(lambda x: [IDtoLabel_UK[i] for i in x])
testDataUK['ner_tags'] = testDataUK['ner_tags'].apply(lambda x: [LabelToID[i] for i in x])

# Worldwide Data
def load_tsvs(folder_path):
    all_sentences, all_ner_tags, all_regions = [], [], []
    for file_path in glob.glob(os.path.join(folder_path, "*.tsv")):
        region = os.path.basename(file_path).split('_')[0]
        cur_sent, cur_tags = [], []
        with open(file_path, 'r', encoding='utf-8') as f:
            for _ in range(2): 
                try: next(f)
                except: break
            for line in f:
                line = line.strip()
                if not line:
                    if cur_sent:
                        all_sentences.append(list(cur_sent))
                        all_ner_tags.append(list(cur_tags))
                        all_regions.append(region)
                        cur_sent, cur_tags = [], []
                    continue 
                parts = line.split('\t')
                if len(parts) >= 2:
                    cur_sent.append(parts[0].strip())
                    cur_tags.append(parts[1].strip())
            if cur_sent:
                all_sentences.append(list(cur_sent))
                all_ner_tags.append(list(cur_tags))
                all_regions.append(region)
    return pd.DataFrame({'tokens': all_sentences, 'ner_tags': all_ner_tags, 'region': all_regions})

dataGlobal = load_tsvs(r"./processed_annotated")
dataGlobal['ner_tags'] = dataGlobal['ner_tags'].apply(lambda x: [LabelToID.get(i, 0) for i in x])
train_ww, test_ww = train_test_split(dataGlobal, test_size=0.2, random_state=42)


def align_entities_to_iob2(tokens, extracted_entities):
    """Maps the LLM JSON entities back to the exact token array using IOB2 tags."""
    # Convert tokens to a standard list to prevent NumPy array truth value errors
    tokens = list(tokens)
    
    iob2_tags = ['O'] * len(tokens)
    for ent in extracted_entities:
        ent_text = str(ent.get('entity', '')).split()
        label = str(ent.get('label', ''))
        
        if not ent_text or not label: continue
        
        ent_len = len(ent_text)
        for i in range(len(tokens) - ent_len + 1):
            if tokens[i:i+ent_len] == ent_text:
                iob2_tags[i] = f"B-{label}"
                for j in range(1, ent_len):
                    if i+j < len(iob2_tags):
                        iob2_tags[i+j] = f"I-{label}"
                break 
    return iob2_tags

def run_llm_evaluation(df_test, dataset_name):
    print(f"\n--- Starting Llama 3.1 on {dataset_name} ---")
    df_sample = df_test.sample(n=min(SAMPLE_SIZE, len(df_test)), random_state=42).reset_index(drop=True)
    
    all_true, all_pred = [], []
    
    for index, row in df_sample.iterrows():
        tokens = list(row['tokens'])
        sentence = " ".join(tokens)
        true_tags = [id2label[tag_id] for tag_id in row['ner_tags']] 
        
        prompt = f"""
        You are an expert Named Entity Recognition system.
        Extract entities from the following sentence. 
        Allowed labels: 'Person', 'Location', 'Organization', 'Misc'.
        
        Sentence: "{sentence}"
        
        Return ONLY a valid JSON list of dictionaries. No markdown, no explanations.
        Format: [{{"entity": "text", "label": "Category"}}]
        """
        
        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant", 
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0 # Strict, zero-creativity parameter
            )
            
            raw_output = response.choices[0].message.content.strip()
            
            # Stripping Markdown formatting if the AI adds it
            if raw_output.startswith("```json"): 
                raw_output = raw_output[7:-3].strip()
            elif raw_output.startswith("```"): 
                raw_output = raw_output[3:-3].strip()
                
            extracted = json.loads(raw_output)
            
        except Exception as e:
            extracted = [] 
            
        pred_tags = align_entities_to_iob2(tokens, extracted)
        
        all_true.extend(true_tags)
        all_pred.extend(pred_tags)
        
        if (index + 1) % 50 == 0: 
            print(f"Processed {index + 1}/{len(df_sample)}...")
            
        time.sleep(0.5) 
        
    score = f1_score(all_true, all_pred, average='macro')
    print(f"Final F1 Score for {dataset_name}: {score:.4f}")
    return score

f1_ww = run_llm_evaluation(test_ww, "Worldwide Test Set")
f1_uk = run_llm_evaluation(testDataUK, "English/UK Test Set")

print("ALL DONE")

Loading data... (this might take a few seconds)

--- Starting Llama 3.1 on Worldwide Test Set ---
Processed 50/200...
Processed 100/200...
Processed 150/200...
Processed 200/200...
Final F1 Score for Worldwide Test Set: 0.3188

--- Starting Llama 3.1 on English/UK Test Set ---
Processed 50/200...
Processed 100/200...
Processed 150/200...
Processed 200/200...
Final F1 Score for English/UK Test Set: 0.6478
ALL DONE
